<a href="https://colab.research.google.com/github/charlyacha/labo1-colabs/blob/main/02_Estadistica_de_una_variable.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 02 — Estadística de una variable: promedio, dispersión y error de la media**Laboratorio 1 · Clase 2****Objetivos.**1. Entender el promedio como **estimador** del valor esperado de una distribución, y bajo qué   supuestos es el mejor estimador disponible.2. Distinguir con precisión la **desviación estándar** $s$ del **error estándar de la media** (SEM).3. Calcular ambas en Python **sin el error silencioso más frecuente de NumPy**.4. Construir un histograma con un criterio objetivo de ancho de bin.**Requisitos previos:** Colab 01.Ésta es la clase conceptualmente más importante del curso. Si algo de acá queda flojo, arrastrahasta la última práctica.> **Antes de empezar:** hacé `Archivo → Guardar una copia en Drive`. Vas a trabajar sobre *tu* copia; el original queda intacto para el resto del curso.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.random.seed(20260819)   # para que todos vean los mismos números de ejemplo

---## 1. Los datosTrabajamos con las mediciones de **tiempo de reacción**. Si todavía no tenés las tuyas, la celda deabajo genera un conjunto de ejemplo con la misma estructura: un valor por línea, en segundos.Cuando tengas tus datos reales, reemplazá esa celda por `t = np.loadtxt('tus_datos.txt')`.

In [ ]:
# --- DATOS DE EJEMPLO (reemplazar por los propios) ---------------------N_ej = 120t = np.random.normal(loc=0.235, scale=0.032, size=N_ej)   # segundosnp.savetxt('tiempos_reaccion.txt', t, fmt='%.4f')# ----------------------------------------------------------------------t = np.loadtxt('tiempos_reaccion.txt')print(f"Se cargaron {len(t)} mediciones.")print("Primeras cinco:", np.round(t[:5], 4), "s")

---## 2. El promedio no es una receta: es un estimadorEscribimos$$\\bar{x} = \\frac{1}{N}\\sum_{i=1}^{N} x_i$$y lo llamamos "el promedio". Pero la pregunta relevante no es cómo se calcula sino **qué estima**.Detrás de tus $N$ mediciones hay una distribución de probabilidad: el proceso de medir tiene unvalor esperado $\\mu$ y una dispersión $\\sigma$. Cada medición individual es una muestra de esadistribución. El promedio es un **estimador de $\\mu$**, y bajo tres supuestos es el mejor que existe:1. los errores son **puramente aleatorios** (no hay sesgo sistemático);2. las mediciones son **independientes** entre sí;3. la distribución tiene **varianza finita**.Si el supuesto 1 falla, el promedio converge — pero converge al valor equivocado. Y como ladispersión sigue siendo chica, el resultado parece excelente. Es *preciso pero sesgado*. Vamos avolver sobre esto en el Colab 04; por ahora quedate con que promediar no arregla un instrumentodescalibrado.

In [ ]:
N = len(t)promedio = np.mean(t)promedio_a_mano = np.sum(t) / Nprint(f"np.mean       : {promedio:.6f} s")print(f"suma/N        : {promedio_a_mano:.6f} s")print(f"¿coinciden?   : {np.isclose(promedio, promedio_a_mano)}")

---## 3. Desviación estándar: el error silencioso de NumPyLa desviación estándar **muestral** es$$s = \\sqrt{\\frac{1}{N-1}\\sum_{i=1}^{N}(x_i - \\bar{x})^2}$$Prestá atención al $N-1$ (**corrección de Bessel**). El motivo: para calcular la dispersión respectode $\\bar{x}$ ya "gastaste" un grado de libertad estimando $\\bar{x}$ a partir de los mismos datos.Dividir por $N$ subestima sistemáticamente la dispersión.**El problema:** `np.std()` usa por defecto `ddof=0`, es decir, divide por $N$. Calcula la desviaciónestándar *poblacional*, que no es la que corresponde a un conjunto de mediciones experimentales.No da error. No avisa. Simplemente devuelve un número un poco chico. **Siempre escribí `ddof=1`.**

In [ ]:
s_mal  = np.std(t)            # ddof=0 por defecto  → POBLACIONAL (incorrecto acá)s_bien = np.std(t, ddof=1)    # muestral            → correctoprint(f"np.std(t)          = {s_mal:.6f} s   <- NO usar")print(f"np.std(t, ddof=1)  = {s_bien:.6f} s   <- usar ésta")print(f"diferencia relativa: {100*(s_bien-s_mal)/s_bien:.3f} %")

In [ ]:
# ¿Cuánto importa? Depende de N. Con N chico, muchísimo.for n in [3, 5, 10, 30, 100, 1000]:    factor = np.sqrt(n / (n - 1))    print(f"N = {n:5d}  ->  s_muestral / s_poblacional = {factor:.4f}"          f"   ({100*(factor-1):.2f} % de subestimación si te olvidás)")

Con $N=3$ —el caso típico de "medí tres veces"— olvidarse de `ddof=1` subestima la dispersión en un22 %. Con $N=1000$ el efecto es despreciable, pero para entonces ya instalaste la idea equivocada.

---## 4. Desviación estándar vs. error estándar de la mediaSon dos cosas distintas y responden a dos preguntas distintas:| Cantidad | Fórmula | Responde a ||---|---|---|| Desviación estándar $s$ | $\\sqrt{\\frac{1}{N-1}\\sum(x_i-\\bar{x})^2}$ | ¿Cuánto se dispersa **una medición individual**? || Error estándar de la media | $\\mathrm{SEM} = s/\\sqrt{N}$ | ¿Cuánta incerteza tiene **el promedio**? |Y de ahí sale la regla que vas a usar todo el cuatrimestre:> **La desviación estándar no disminuye al aumentar $N$** — es una propiedad del proceso de medición.> **El error de la media sí disminuye**, como $1/\\sqrt{N}$ — porque promediar más mediciones mejora la> estimación del valor central, no la calidad de cada medición individual.Cuál de las dos reportás depende de qué estés informando. Si el resultado de tu experimento es elpromedio, la incerteza que corresponde es el SEM. Si querés describir la variabilidad del proceso(por ejemplo, cuánto varía tu tiempo de reacción entre intentos), lo que corresponde es $s$.

In [ ]:
sem = s_bien / np.sqrt(N)print(f"N   = {N}")print(f"x̄   = {promedio:.5f} s")print(f"s   = {s_bien:.5f} s      (dispersión de una medición individual)")print(f"SEM = {sem:.5f} s      (incerteza del promedio)")print(f"\nRelación s/SEM = √N = {s_bien/sem:.2f}  (y √N = {np.sqrt(N):.2f})")

### Verificación empíricaEn vez de creerle a la fórmula, la miramos. Calculamos $s$ y SEM usando los primeros $n$ datos, con$n$ creciendo de 2 hasta $N$, y graficamos las dos curvas.

In [ ]:
ns = np.arange(2, N + 1)s_acum   = np.array([np.std(t[:n], ddof=1) for n in ns])sem_acum = s_acum / np.sqrt(ns)fig, ax = plt.subplots(figsize=(7, 4.2))ax.plot(ns, s_acum,   'o-', ms=3, lw=1, label='$s$  (desviación estándar)')ax.plot(ns, sem_acum, 's-', ms=3, lw=1, label='SEM $= s/\\sqrt{N}$')ax.plot(ns, s_bien / np.sqrt(ns), 'k--', lw=1, label='$s_{final}/\\sqrt{N}$ (teórico)')ax.set_xlabel('Cantidad de mediciones usadas $N$')ax.set_ylabel('Dispersión [s]')ax.set_title('La desviación estándar se estabiliza; el error de la media cae como $1/\\sqrt{N}$')ax.grid(alpha=0.3); ax.legend()fig.tight_layout(); plt.show()

Mirá el gráfico con atención: la curva de $s$ fluctúa al principio y después **se aplana**. La delSEM **sigue bajando**. Ésa es toda la diferencia entre las dos cantidades, en una figura.> **Ejercicio 2.1.** ¿Cuántas mediciones necesitarías para reducir el SEM a la mitad del valor que> tiene ahora? ¿Y a la décima parte? ¿Te parece razonable el costo experimental?

---## 5. El histograma y el ancho de binEl histograma es la primera mirada a la forma de la distribución. Pero el resultado depende delancho de bin elegido, y ése es un problema real: con bins muy anchos toda distribución parece unasola barra; con bins muy finos, ruido.Elegir "hasta que se vea lindo" es una decisión arbitraria que puede cambiar la conclusión. Existencriterios objetivos. El más simple es la **regla de Scott** (Scott, *Biometrika* **66**(3), 605-610,1979):$$h = \\frac{3{,}49\\, s}{N^{1/3}}$$que minimiza el error cuadrático medio integrado suponiendo que la distribución subyacente esaproximadamente normal. Una alternativa más robusta frente a valores atípicos es la de**Freedman–Diaconis**, que usa el rango intercuartílico en lugar de $s$.

In [ ]:
def bins_scott(x):    """Cantidad de bins según la regla de Scott."""    x = np.asarray(x)    n = len(x)    h = 3.49 * np.std(x, ddof=1) / n**(1/3)    return max(1, int(np.ceil((x.max() - x.min()) / h)))def bins_freedman_diaconis(x):    """Cantidad de bins según Freedman-Diaconis (más robusta a outliers)."""    x = np.asarray(x)    n = len(x)    iqr = np.percentile(x, 75) - np.percentile(x, 25)    h = 2 * iqr / n**(1/3)    return max(1, int(np.ceil((x.max() - x.min()) / h)))print("Scott            :", bins_scott(t), "bins")print("Freedman-Diaconis:", bins_freedman_diaconis(t), "bins")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=False)for ax, nb, titulo in zip(axes,                          [4, bins_scott(t), 60],                          ['4 bins (demasiado grueso)',                           f'{bins_scott(t)} bins (regla de Scott)',                           '60 bins (demasiado fino)']):    ax.hist(t, bins=nb, edgecolor='k', alpha=0.75)    ax.set_title(titulo, fontsize=10)    ax.set_xlabel('Tiempo de reacción [s]')    ax.grid(alpha=0.3)axes[0].set_ylabel('Frecuencia')fig.tight_layout(); plt.show()

Los tres histogramas son de **los mismos datos**. El primero no te deja ver si la distribución essimétrica; el tercero te hace ver estructura que no existe. Por eso el criterio de bin se declara enel informe, igual que cualquier otra decisión de análisis.

In [ ]:
# El histograma final, con las estadísticas marcadasfig, ax = plt.subplots(figsize=(7, 4.2))ax.hist(t, bins=bins_scott(t), edgecolor='k', alpha=0.7)ax.axvline(promedio, color='crimson', lw=2, label=f'$\\bar{{x}}$ = {promedio:.4f} s')ax.axvspan(promedio - s_bien, promedio + s_bien, color='crimson', alpha=0.12,           label=f'$\\bar{{x}} \\pm s$  (s = {s_bien:.4f} s)')ax.set_xlabel('Tiempo de reacción [s]')ax.set_ylabel('Frecuencia')ax.set_title(f'Tiempo de reacción — N = {N}')ax.grid(alpha=0.3); ax.legend()fig.tight_layout(); plt.show()

---## 6. El resultado

In [ ]:
from math import floor, log10def formatear(x, dx, unidad=""):    orden = floor(log10(abs(dx)))    cifras = 2 if int(dx / 10**orden) == 1 else 1    dec = max(-(orden - (cifras - 1)), 0)    return f"({x:.{dec}f} ± {dx:.{dec}f}) {unidad}".strip()print("Si informás el valor central de tu tiempo de reacción:")print("   t =", formatear(promedio, sem, "s"), "  (incerteza = SEM)")print()print("Si informás cuánto varía tu tiempo de reacción entre intentos:")print("   t =", formatear(promedio, s_bien, "s"), "  (incerteza = s)")

---## 7. Ejercicios**2.2.** Con tus propios datos, calculá $\\bar{x}$, $s$ y SEM. Escribí en una oración cuál de los dosreportás como incerteza de tu resultado y por qué. (Ésta es la respuesta que se pide en la Entregacorta 1.)**2.3.** Dividí tus datos en dos mitades y calculá $\\bar{x}$ y SEM para cada una. ¿Son compatiblesentre sí? Si no lo son, ¿qué podría estar pasando? (Pista: ¿te fuiste cansando?)**2.4.** Repetí el gráfico de la sección 4 pero con los datos **mezclados al azar**(`np.random.permutation(t)`). ¿Cambia la curva de $s$? ¿Y la de SEM? ¿Qué te dice eso sobre elsupuesto de independencia?**2.5.** Auditá cualquier código de análisis que ya tengas (tuyo o de un compañero) buscando`np.std(` sin `ddof=1`. Es la clase de error que sobrevive años sin que nadie lo note.